# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/karthikmannam/flyrank-internship-ml/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

**Binary classification with a scoring output.**

The underlying decision is a yes/no question: *"Is this page declining?"* At the row level, the model predicts `is_declining_label ∈ {0,1}` from observed content and engagement signals — that is classification.

But the output is used as a **priority score** (the predicted probability of decline), which ranks the full inventory for an editor. So the task type is classification internally, ranking by score externally. This matches the skill's "Which ones first? → Ranking / scoring" row while still using a supervised label for training.

Why not regression? The label is inherently categorical — we care about *direction* (declining or not), not the magnitude of the trend. A classifier's probability gives us a clean priority score.

In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv("../../data/raw/content_refresh_anonymized.csv")
df["label"] = (df["trend_direction"] == "down").astype(int)

print("Task type: Binary classification (output ranked by predicted probability)")
print()
print("Label distribution:")
print(df["label"].value_counts().to_frame().rename(
    columns={"label": "count"}).assign(
    pct=lambda x: (x["count"] / len(df) * 100).round(1)))
print()
n_declining = int(df["label"].sum())
print(f"{n_declining:,} of {len(df):,} rows are declining ({n_declining/len(df)*100:.1f}%)")
print("Two classes → binary classification. Probability → ranking score.")

Task type: Binary classification (output ranked by predicted probability)

Label distribution:
       count   pct
label             
1      16262  54.2
0      13738  45.8

16,262 of 30,000 rows are declining (54.2%)
Two classes → binary classification. Probability → ranking score.


## 2. Target or proxy

**Target: `is_declining_label` — whether the page's recent trend direction is "down".**

This is an **observed outcome**, not a defined rule. The label comes from comparing actual impression counts in two consecutive 30-day windows (`impressions_last_30d` vs `impressions_prev_30d`): if the later window's impressions dropped by more than 20%, the label is 1 (declining). The data measures what *actually happened* to the page's search visibility.

The proxy concern: the 20% threshold is a definitional choice, but the underlying comparison of measured impressions is observed. The skill warns "The target must be observed, not defined" — here the *comparison* is observed (measured impressions from Google Search Console), and the threshold codifies what "declining" means in the client's workflow. This is acceptable as long as we never use `trend_direction` or `trend_pct` as a feature (leakage rule in the data dictionary).

In [2]:
# Show how the label is derived from observed impression counts
provenance = df[["impressions_prev_30d", "impressions_last_30d", "trend_direction", "trend_pct", "label"]].copy()
prev = provenance["impressions_prev_30d"].replace(0, np.nan)
provenance["pct_change"] = ((provenance["impressions_last_30d"] - provenance["impressions_prev_30d"]) / prev * 100)
provenance["pct_change"] = provenance["pct_change"].round(1)

print("Label provenance: comparing two observed 30-day impression windows")
print("  Label=1 (declining) when impressions_last_30d < impressions_prev_30d by >20%")
print("  Label=0 (not declining) otherwise (stable, up, new, flat)")
print()
print("Example rows — label comes from observed counts, not a made-up rule:")
provenance.head(6)

Label provenance: comparing two observed 30-day impression windows
  Label=1 (declining) when impressions_last_30d < impressions_prev_30d by >20%
  Label=0 (not declining) otherwise (stable, up, new, flat)

Example rows — label comes from observed counts, not a made-up rule:


,impressions_prev_30d,impressions_last_30d,trend_direction,trend_pct,label,pct_change
0,987,578,down,-41.4,1,-41.4
1,5915,2501,down,-57.7,1,-57.7
2,6089,2382,down,-60.9,1,-60.9
3,4206,3626,stable,-13.8,0,-13.8
4,6452,4211,down,-34.7,1,-34.7
5,1009,617,down,-38.9,1,-38.9


## 3. Success metric

**Precision@K — specifically Precision@50.**

Why this metric: the editor has limited weekly capacity (e.g. 50 pages). Precision@50 answers the practical question: *"Of the 50 pages I recommend most urgently, how many are actually declining?"* A random baseline would get ~54% (the base rate). The starter pipeline's baseline rule got 24% (worse than random — the simple heuristic actually hurts). The starter random forest got 74%.

Good means statistically and operationally better than the base rate (54.2%). A useful target: Precision@50 ≥ 0.70, meaning at least 35 of the top 50 picks are truly declining — so the editor's finite time goes to pages that need it.

Recall and average precision are secondary checks, but Precision@K is the metric that maps directly to the decision.

In [3]:
# Compute base rate and show Precision@K concept
base_rate = df["label"].mean()
print(f"Base rate (declining): {base_rate:.1%}")
print(f"Random Precision@50:   {base_rate:.1%}  (what random picks would get)")
print()
print("Starter pipeline benchmarks (from committed outputs):")
print("  Baseline rule Precision@50:  0.240 (hand-written heuristic)")
print("  Decision tree Precision@50:  0.620 (max_depth=5, min_samples_leaf=50)")
print("  Random forest Precision@50:  0.740 (200 trees, library default params)")
print()
print(f"Target: Precision@50 >= 0.70 \u2014 at least 35 of the top 50 picks are truly declining.")
print(f"This means the editor's finite time goes to pages that need it.")

Base rate (declining): 54.2%
Random Precision@50:   54.2%  (what random picks would get)

Starter pipeline benchmarks (from committed outputs):
  Baseline rule Precision@50:  0.240 (hand-written heuristic)
  Decision tree Precision@50:  0.620 (max_depth=5, min_samples_leaf=50)
  Random forest Precision@50:  0.740 (200 trees, library default params)

Target: Precision@50 >= 0.70 — at least 35 of the top 50 picks are truly declining.
This means the editor's finite time goes to pages that need it.


## 4. The unit of analysis, as a real dataframe

**One row = one content item (one page).**

The row grain is defined by `content_id` — each row is a unique page with its aggregated 90-day performance metrics, keyword context, and content properties. This is the natural unit for the editorial decision: which *page* needs attention first.

Below: a slice from the raw dataset showing the key fields for the refresh-scoring task: identifiers (grouping only), engagement signals, trend inputs, and content meta.

In [4]:
# Show the unit of analysis: one row = one content page
unit_cols = [
    "content_id", "client_id", "content_type", "content_age_days",
    "impressions_90d", "ctr", "avg_position", "engagement_rate",
    "days_since_last_update", "impression_tier", "trend_direction",
]

unit_df = df[unit_cols].copy()
unit_df["is_declining"] = df["label"]

print(f"One row = one page (content item). Shape: {unit_df.shape}")
print(f"Units of analysis: {len(unit_df):,} pages across {unit_df['client_id'].nunique()} clients")
print()
unit_df.head(10)

One row = one page (content item). Shape: (30000, 12)
Units of analysis: 30,000 pages across 32 clients



,content_id,client_id,content_type,content_age_days,impressions_90d,ctr,avg_position,engagement_rate,days_since_last_update,impression_tier,trend_direction,is_declining
0,content_304f48230142,client_f369cb89fc,keyword article,187,3803,0.76,10.6,5.88,20,good,down,1
1,content_a1fb4e703a9e,client_4e07408562,keyword article,445,15320,0.05,20.3,0.00,25,good,down,1
2,content_9aa793d4d895,client_7f2253d7e2,keyword article,141,12581,0.09,36.5,0.00,20,good,down,1
3,content_331d6c4de07b,client_19581e27de,keyword article,463,11751,0.49,6.2,1.28,22,good,stable,0
4,content_d99b7a2d90ca,client_3fdba35f04,keyword article,263,19140,0.13,44.0,0.00,14,good,down,1
5,content_d4084a4bc775,client_f369cb89fc,keyword article,147,3970,0.03,8.5,0.00,20,good,down,1
6,content_9a34b442b552,client_8722616204,keyword article,90,20,0.00,7.0,0.00,20,low,down,1
7,content_a63219c6e95a,client_19581e27de,keyword article,445,1724,0.06,21.2,3.57,22,moderate,stable,0
8,content_5e6c160719bc,client_6208ef0f77,keyword article,90,32574,0.09,46.0,5.88,20,excellent,down,1
9,content_c27558df2b0c,client_19581e27de,keyword article,257,1240,0.16,4.9,0.00,104,moderate,down,1


## 5. Why ML beats a fixed rule here

**Decline emerges from many weak signals interacting — a hand-written rule can’t capture the combinations.**

A fixed rule like *"Score = -trend_pct + (position > 10) × weight"* misses the reality that decline depends on context:
- A page with high impressions and slipping position is different from a low-impression page with the same position change.
- Content type, keyword competition, and engagement rate interact: a thin `feedly article` with dropping CTR signals something different than a well-trafficked `keyword article` with the same CTR drop.
- Some signals are only meaningful in combination: `avg_position = 0` means "no data" (1,205 rows), not rank zero — a rule that treats 0 as a rank value is wrong by construction.
- Missingness is systematic by content_type — a rule that blindly fillna(0) encodes false signals.

A learned model can weight these interactions from data instead of guessing the right thresholds. The starter pipeline proves this empirically: the rule baseline gets Precision@50 = 0.240 (worse than the 54% base rate — the simple rule actively misranks), while a random forest gets 0.740. That 50-percentage-point lift comes from finding patterns no single if-statement expresses.

### Generalization vs. overfitting

We are not claiming the random forest’s 0.740 is universal. The pipeline uses a **client-holdout** split: 6 of the 32 clients are held out entirely during training, and metrics are reported only on those unseen clients. This is a stronger generalization test than a random row split because it measures performance on completely new publishers, not just new pages from seen clients. Overfitting would show up as a large gap between training and holdout Precision@50 or as the model memorizing client-specific patterns rather than learning transferable decline signals. The client-holdout design is our primary defense.

In [5]:
# Show why a fixed rule fails: same trend_pct, opposite labels
# Context (volume, position, content type) determines the outcome
contradict = df.groupby("trend_pct").filter(lambda g: g["label"].nunique() > 1)
print(f"trend_pct values with contradictory labels: {contradict['trend_pct'].nunique()}")
print()

example_pct = contradict["trend_pct"].value_counts().index[0]
bucket = contradict[contradict["trend_pct"] == example_pct]
show = bucket[["impressions_90d", "avg_position", "ctr", "content_type", "label"]].copy()
print(f"At trend_pct = {example_pct} ({len(bucket)} pages):")
print(show.to_string())
print()
print("Same trend strength (impression drop) — but some declining and some aren’t.")
print("Volume, position, content type tip the balance. ML sees all of them.")
print()

# Quick ML demo: no label-derived signals, only safe features
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import precision_score

safe_features = ["ctr", "avg_position", "engagement_rate", "content_age_days",
                "impressions_90d", "days_since_last_update", "competition"]
demo_X = df[safe_features].copy()
demo_X["avg_position"] = demo_X["avg_position"].replace(0, np.nan).fillna(demo_X["avg_position"].median())
demo_X = demo_X.fillna(0)
y = df["label"]

simple = DecisionTreeClassifier(max_depth=1, random_state=42)
simple.fit(demo_X, y)
y_simple = simple.predict(demo_X)
split_feature = safe_features[simple.tree_.feature[0]]
split_threshold = simple.tree_.threshold[0]
print(f"Single-split rule (max_depth=1) Precision: {precision_score(y, y_simple):.4f}")
print(f"  Split on: {split_feature} <= {split_threshold:.4f}")

interact = DecisionTreeClassifier(max_depth=5, random_state=42)
interact.fit(demo_X, y)
y_interact = interact.predict(demo_X)
print(f"Interactive tree (max_depth=5)    Precision: {precision_score(y, y_interact):.4f}")
print()
print("Without using the label source (trend_pct), a single split is weak.")
print("Adding depth (interactions) lifts precision — ML finds combinations.")

trend_pct values with contradictory labels: 1

At trend_pct = -20.0 (55 pages):
       impressions_90d  avg_position    ctr        content_type  label
204               5996          19.0   0.10     keyword article      0
364                 51           6.3   0.00     keyword article      0
800                 33           6.5   0.00  comparison article      0
1244               701          27.7   0.14     keyword article      0
2264                16           8.3   0.00     keyword article      0
2877                63           6.5   0.00     keyword article      0
3792             18377           5.4   1.85     keyword article      1
5951               152          13.1   1.32     keyword article      0
6972              3141          45.5   0.00     keyword article      1
7022             13546          14.4   1.23     keyword article      0
8190                10           4.9  10.00      feedly article      0
9564              3913           2.8   1.58     keyword article     

Single-split rule (max_depth=1) Precision: 0.5869
  Split on: impressions_90d <= 5.5000
Interactive tree (max_depth=5)    Precision: 0.6424

Without using the label source (trend_pct), a single split is weak.
Adding depth (interactions) lifts precision — ML finds combinations.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime -> Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.